In [1]:
#pip install mlxtend --upgrade

In [2]:
import ast
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

In [3]:
basket = pd.read_csv('customer_basket.csv')
clusters = pd.read_csv('cluster_assignments.csv')
basket = basket.merge(clusters, on='customer_id')

print(f"Total baskets: {len(basket)}")
print(f"Baskets per cluster:\n{basket['cluster'].value_counts().sort_index()}")

Total baskets: 100000
Baskets per cluster:
cluster
0     9488
1    31595
2    37892
3    14505
4     6520
Name: count, dtype: int64


In [4]:
basket['list_of_goods'] = basket['list_of_goods'].apply(ast.literal_eval)

# Quick sanity check
print(f"Sample basket: {basket['list_of_goods'].iloc[0]}")
print(f"Sample cluster: {basket['cluster'].iloc[0]}")

Sample basket: ['chicken', 'rice', 'pepper', 'whole wheat rice', 'shrimp', 'toothpaste', 'gums', 'cereals', 'bluetooth headphones', 'vacuum cleaner', 'body spray']
Sample cluster: 1


In [5]:
def get_association_rules(transactions, min_support=0.02, min_confidence=0.2):
    te = TransactionEncoder()
    te_array = te.fit_transform(transactions)
    df = pd.DataFrame(te_array, columns=te.columns_)
    frequent_itemsets = apriori(df, min_support=min_support, use_colnames=True)
    if frequent_itemsets.empty:
        print("No frequent itemsets found")
        return pd.DataFrame()
    rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)
    return rules.sort_values('lift', ascending=False)

In [6]:
all_products = [item for sublist in basket['list_of_goods'] for item in sublist]
print(f"Unique products: {len(set(all_products))}")
print(f"Most common products:\n{pd.Series(all_products).value_counts().head(10)}")

Unique products: 164
Most common products:
asparagus       12811
airpods         12145
cereals          9952
fresh bread      9934
butter           9654
eggs             9241
protein bar      8695
cooking oil      8623
toilet paper     8395
babies food      8318
Name: count, dtype: int64


In [7]:
print(basket['cluster'].value_counts().sort_index())
print(basket['cluster'].isna().sum())

cluster
0     9488
1    31595
2    37892
3    14505
4     6520
Name: count, dtype: int64
0


In [8]:
print(basket[basket['cluster'].isna()].shape)
basket = basket.dropna(subset=['cluster'])
basket['cluster'] = basket['cluster'].astype(int)
print(basket['cluster'].value_counts().sort_index())

(0, 4)
cluster
0     9488
1    31595
2    37892
3    14505
4     6520
Name: count, dtype: int64


In [9]:
# 4. Run per cluster and store results
cluster_rules = {}
for cluster_id in sorted(basket['cluster'].unique()):
    transactions = basket[basket['cluster'] == cluster_id]['list_of_goods'].tolist()
    rules = get_association_rules(transactions)
    cluster_rules[cluster_id] = rules

In [10]:
for cluster_id in sorted(basket['cluster'].unique()):
    print(f"\n{'='*50}")
    print(f"CLUSTER {cluster_id} — Top Unique Rules by Lift")
    print(f"{'='*50}")
    rules = cluster_rules[cluster_id]
    if not rules.empty:
        # Convert frozensets to sorted tuples to identify duplicates
        rules = rules.copy()
        rules['pair'] = rules.apply(
            lambda row: tuple(sorted([
                str(sorted(row['antecedents'])), 
                str(sorted(row['consequents']))
            ])), axis=1
        )
        # Keep only one direction per pair
        unique_rules = rules.drop_duplicates(subset='pair')
        top10 = unique_rules.nlargest(10, 'lift')[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
        display(top10)


CLUSTER 0 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
579,(cereals),"(eggs, green tea)",0.021185,0.059982,1.368053
581,(green tea),"(cereals, eggs)",0.021185,0.196673,1.361076
577,"(cereals, green tea)",(eggs),0.021185,0.462069,1.349372
540,"(oatmeal, butter)",(milk),0.020341,0.260811,1.344146
609,(cereals),"(eggs, salt)",0.026454,0.074903,1.338380
604,(eggs),"(cereals, oil)",0.021606,0.063096,1.330351
774,"(honey, fresh bread)",(tea),0.022028,0.272846,1.330299
676,(honey),"(cereals, oatmeal)",0.022344,0.109959,1.329027
688,(oatmeal),"(cereals, milk)",0.021185,0.103502,1.328853
677,(oatmeal),"(cereals, honey)",0.022344,0.109166,1.322816



CLUSTER 1 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
11,(deodorant),(shower gel),0.024877,0.283652,3.445593
16,(shower gel),(shampoo),0.024782,0.301038,3.441135
23,(tooth brush),(shower gel),0.024276,0.274026,3.328668
9,(shampoo),(deodorant),0.024846,0.284009,3.238273
19,(shampoo),(tooth brush),0.024687,0.282200,3.185459
25,(toothpaste),(shower gel),0.024687,0.252672,3.069276
12,(deodorant),(tooth brush),0.023675,0.269939,3.047057
26,(tooth brush),(toothpaste),0.025859,0.291890,2.987452
21,(shampoo),(toothpaste),0.025162,0.287627,2.943817
14,(deodorant),(toothpaste),0.024403,0.278239,2.847735



CLUSTER 2 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
3,(airpods),(energy drink),0.023884,0.200044,2.612018
13,(napkins),(cooking oil),0.020295,0.228733,2.583352
0,(bluetooth headphones),(airpods),0.022327,0.297468,2.491528
14,(dog food),(napkins),0.022274,0.215581,2.429744
5,(babies food),(cooking oil),0.021720,0.214100,2.418084
9,(napkins),(babies food),0.021588,0.243308,2.398390
7,(babies food),(dog food),0.023963,0.236212,2.286221
11,(dog food),(cooking oil),0.020506,0.198467,2.241528



CLUSTER 3 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
178,"(asparagus, carrots)",(salad),0.027646,0.438251,3.555277
163,(salad),"(avocado, asparagus)",0.026680,0.216443,3.511751
161,(avocado),"(asparagus, salad)",0.026680,0.226979,3.465618
154,"(asparagus, carrots)",(avocado),0.025577,0.405464,3.449421
199,(salad),"(asparagus, spinach)",0.027094,0.219799,3.409818
166,"(asparagus, spinach)",(avocado),0.025440,0.394652,3.357439
205,(salad),"(tomatoes, asparagus)",0.026405,0.214206,3.326612
173,(avocado),"(tomatoes, asparagus)",0.024957,0.212317,3.297274
183,"(asparagus, carrots)",(spinach),0.027508,0.436066,3.278969
78,(energy drink),(bluetooth headphones),0.025577,0.280423,3.269726



CLUSTER 4 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
0,(asparagus),(airpods),0.021779,0.137597,1.059187


**Support** - proportion of baskets that contain both items

**Confidence** - percentage of bought together ('If someone buys item A, we have a x% confidence that buys item B)

**Lift** - measure of how much more likely someone is to buy two items together compared to independently
- Lift = 1 → no association, purely coincidental
- Lift > 1 → positive association, bought together more than expected
- Lift < 1 → negative association, rarely bought together

# Cluster Campaigns

**Cluster 0 - "Big Family Shopper"**
- eggs + green tea → cereals (lift 1.37), cereals + eggs → green tea (lift 1.36), honey → cereals + outmeal (lift 1.32) - Strong healthy breakfast pattern
- high drinks (alcohol and non-alcohol) and hygiene spending
- Campaign: **"Family Breakfast Bundle - buy cereals + green tea + eggs and get honey and outmeal 30% off"**, **"Party Ready Bundle - buy 3 alcoholic drinks and get 20% off non-alcoholic drinks"** and **"Family Hygiene Pack - buy €30 in hygiene products and get the cheapest one free"**

**Cluster 1 - "New Customers + Tech Enthusiast"**
- Surprisingly dominated by hygiene products - shower gel → shampoo (lift 3.44), deodorant → shower gel (lift 3.44), tooth brush → shower gel (lift 3.33), shampoo → tooth brush (lift 3.19)
- high fresh_food_ratio and tech spending 
- Campaign: **"Hygiene Kit - buy shampoo + shower gel + deodorant and get a toothbrush free"**, **"Fresh Tech Deal - buy €50 in fresh food and get 10% off any electronics"** and **"Welcome to the Club - sign up for loyalty card and get 15% off in Hygiene/Tech products"** (category changes every week)

**Cluster 2 - "Most Loyal"**
- energy drink → airpods (lift 2.61), bluetooth headphones → airpods (lift 2.49)
- high fresh_food_ratio, hygiene and tech spending 
- has_loyalty_card is 1
- Campaign: **"Tech Bundle - buy AirPods and get a free Energy Drink"** or **"Loyal Member Reward - use your loyalty card and get 10% off everything from the category Fresh Food/Hygiene/Tech"** (category changes every week)

**Cluster 3 - "Bargain Hunter"**
- asparagus + carrots → salad (lift 3.56), salad → asparagus + avocado (lift 3.52), asparagus + carrots → avocado (lift 3.45), spinach → asparagus + carrots (lift 3.28), bluetooth headphones → energy drink (lift 3.27)
- spend mostly on sales
- Campaign: **"Healthy Plate Deal - buy asparagus + salad + spinach and get carrots + avocado at half price"** and **"Double Promotion Day — every Wednesday all your usual promoted items get an extra 10% off"**

**Cluster 4 - "Biggest Buyers"**
- Only 1 meaningful rule (asparagus → airpods, lift 1.07) which is very weak
- Don't base the campaign on association rules here - use their spending profile instead
- Campaign: **"VIP Exclusive — spend over €150 and get 15% off your entire basket"**
